In [ ]:
from trainer.distillation import Trainer
from omegaconf import OmegaConf
import torch
from PIL import Image
import numpy as np


from diffusers.utils import make_image_grid

from pathlib import Path

import os


In [ ]:
CONFIG_PATH = Path("/home/rick-mbp/Diffusion-Distillation/configs/qwen_dmd.yaml")

DATASET_PATH = Path(
    "/home/rick-mbp/Diffusion-Distillation/qwen_training_data/qwen_distill_dataset_v1.0.0/metadata_edit.csv"
)


LOG_DIR = "./log"
WANDB = "./wandb"


os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "12355"  # Any free port
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"
os.environ["WORLD_SIZE"] = "1"

In [ ]:
config = OmegaConf.load(CONFIG_PATH)


config.no_save = True
config.no_visualize = True

# get the filename of config_path


config.config_name = CONFIG_PATH.stem
config.logdir = LOG_DIR
config.wandb_save_dir = WANDB
config.disable_wandb = True
config.gradient_checkpointing = True

In [ ]:
trainer = Trainer(config)

In [ ]:
batch = next(trainer.dataloader)
critic_extra = trainer.fwdbwd_one_step(batch, False)

In [ ]:
batch = next(trainer.dataloader)

generator_extra = trainer.fwdbwd_one_step(batch, True)


In [ ]:
generator_extra


In [ ]:
# trainable_params = [p for p in trainer.model.fake_score.parameters() if p.requires_grad]
trainable_params = [p for p in trainer.model.generator.parameters() if p.requires_grad]


In [ ]:
trainable_params[0].grad


In [ ]:
REAL_LORA_NAME = "real"
FAKE_LORA_NAME = "fake"
GENERATOR_LORA_NAME = "generator"


In [ ]:
trainer.model.switch_to_fake()

for name, param in trainer.model.fake_score.transformer.named_parameters():
    # if GENERATOR_LORA_NAME in name or FAKE_LORA_NAME in name:
    if param.requires_grad:
        print(name)
